In [ ]:
import pandas as pd
from pandas import DataFrame
import spacy
from spacy import displacy
from spacy.tokens import DocBin
import json
from datetime import datetime
from tqdm import tqdm
import re
import pickle

My dataprerocessing. Probably not that great, but it is what it is. 

Not documented nicely, sorry, no time D: 

In [ ]:
# Loading train dataset
tags_golden = pd.read_csv('/Data/GutBrainIE_Full_Collection_2026/Annotations/Train/gold_quality/csv_format/train_gold_entities.csv', sep='|')
articles_golden = pd.read_csv('//Data/GutBrainIE_Full_Collection_2026/Articles/csv_format/articles_train_gold.csv', sep = '|')

tags_silver_2025 = pd.read_csv('/Data/GutBrainIE_Full_Collection_2026/Annotations/Train/silver_quality/csv_format/train_silver_2025_entities.csv', sep='|')
articles_silver_2025 = pd.read_csv('/Data/GutBrainIE_Full_Collection_2026/Articles/csv_format/articles_train_silver_2025.csv', sep = '|')

tags_silver = pd.read_csv('/Data/GutBrainIE_Full_Collection_2026/Annotations/Train/silver_quality/csv_format/train_silver_entities.csv', sep='|')
articles_silver = pd.read_csv('/Data/GutBrainIE_Full_Collection_2026/Articles/csv_format/articles_train_silver.csv', sep = '|')

# lines 73918 and 73924 have an extra field and a weird label, skippiing those
tags_bronze = pd.read_csv('/Data/GutBrainIE_Full_Collection_2026/Annotations/Train/bronze_quality/csv_format/train_bronze_entities.csv', sep='|',on_bad_lines='skip')
articles_bronze = pd.read_csv('/Data/GutBrainIE_Full_Collection_2026/Articles/csv_format/articles_train_bronze.csv', sep = '|')



article_col_list = ['pmid', 'title', 'abstract']

articles_title_abstract_golden = articles_golden[article_col_list]
articles_title_abstract_golden['source'] = "golden"
articles_title_abstract_silver_2025 = articles_silver_2025[article_col_list]
articles_title_abstract_silver_2025['source'] = "silver"
articles_title_abstract_silver = articles_silver[article_col_list]
articles_title_abstract_silver['source'] = "silver"
articles_title_abstract_bronze = articles_bronze[article_col_list]
articles_title_abstract_bronze['source'] = "bronze"


# formatting entities
col_list = ['pmid', 'label', 'location', 'start_idx', 'end_idx']

entities_golden = tags_golden[col_list]
entities_silver_2025 = tags_silver_2025[col_list]
entities_silver = tags_silver[col_list]
entities_bronze = tags_bronze[col_list]


# CHECK LATER FOR WEIRD LABELS
list_ent_golden = pd.unique(entities_golden['label'])
list_ent_silver_2025 = pd.unique(entities_silver_2025['label'])
list_ent_silver = pd.unique(entities_silver['label'])
list_ent_bronze = pd.unique(entities_bronze['label'])


articles_full = pd.concat([articles_title_abstract_golden,
                            articles_title_abstract_silver_2025,
                            articles_title_abstract_silver,
                            articles_title_abstract_bronze],
                           ignore_index = True)


entities_full = pd.concat([entities_golden,
                            entities_silver_2025,
                            entities_silver,
                            entities_bronze],
                           ignore_index = True)

In [ ]:
# Loading dev dataset 
tags_dev = pd.read_csv('/Data/GutBrainIE_Full_Collection_2026/Annotations/Dev/csv_format/dev_entities.csv', sep = '|')
articles_dsv = pd.read_csv('/Data/GutBrainIE_Full_Collection_2026/Articles/csv_format/articles_dev.csv', sep = '|')

article_col_list = ['pmid', 'title', 'abstract']
articles_dev = articles_dsv[article_col_list]

col_list = ['pmid', 'label', 'location', 'start_idx', 'end_idx']
entities_dev = tags_dev[col_list]

In [ ]:
def data_preprocessing(entities: DataFrame, articles: DataFrame) -> DataFrame:
    
    """
    Preprocess and merge entity and article DataFrames.

    Parameters
    ----------
    entities : DataFrame
        DataFrame containing entity data: pmid, label, location, start_idx, end_idx
    articles : DataFrame
        DataFrame containing article data: pmid, title, abstract

    Returns
    -------
    DataFrame
        Cleaned and merged DataFrame.
    """
    
    results = {}

    for ent_index, ent_row in entities.iterrows():
        pmid = ent_row.pmid
    
        # Find matching article
        art_row = articles[articles['pmid'] == pmid]
        if art_row.empty:
            continue
        art_row = art_row.iloc[0]
    
        # Initialize entry if first time seeing this pmid
        if pmid not in results:
            results[pmid] = {
                'title': art_row['title'],  
                'title_entities': {},                
                'abstract': art_row['abstract'] if pd.notna(art_row['abstract']) else [],
                'abstract_entities': {}             
                }
    
        # Extract entity info
        entity_info = (ent_row['start_idx'], ent_row['end_idx'], ent_row['label'])
    
        # Add to appropriate dict
        if ent_row['location'] == 'title':
            if results[pmid]['title_entities']:
                new_key = max(results[pmid]['title_entities'].keys()) + 1
            else:
                new_key = 0
            results[pmid]['title_entities'][new_key] = entity_info
        elif ent_row['location'] == 'abstract':
            if results[pmid]['abstract_entities']:
                new_key = max(results[pmid]['abstract_entities'].keys()) + 1
            else:
                new_key = 0
            results[pmid]['abstract_entities'][new_key] = entity_info
        

    proccessed_data = []
    for pmid, entry in results.items():
        # Title
        title_text = entry['title']
        title_entities = [(start, end, label) for key, (start, end, label) in entry['title_entities'].items()]
        proccessed_data.append((title_text, {"entities": title_entities, "source": "title", "pmid": pmid}))
        
        # Abstract
        if entry['abstract']:
            abstract_text = entry['abstract']
            abstract_entities = [(start, end, label) for key, (start, end, label) in entry['abstract_entities'].items()]
            proccessed_data.append((abstract_text, {"entities": abstract_entities, "source": "abstract", "pmid": pmid}))

    return proccessed_data

train_data = data_preprocessing(entities_full, articles_full)
dev_data = data_preprocessing(entities_dev, articles_dev)


In [ ]:
with open("train_data.pkl", "wb") as f:
    pickle.dump(train_data, f)
    
with open("dev_data.pkl", "wb") as f:
    pickle.dump(dev_data, f)